# PPE Detection for Industry 5.0 — Dataset Exploration

**Construction Site Safety** dataset (Roboflow, v30) · YOLOv8 format

This notebook covers Phase 2 of the project:
1. Download the dataset into Colab
2. Inspect classes and train/val/test splits
3. Visualize annotated sample images
4. Analyze class balance

> Run on **Colab with a T4 GPU**: Runtime -> Change runtime type -> T4 GPU.

## 1. Setup

Add your Roboflow API key as a **Colab secret** - do **not** hard-code it:

- Click the key icon in the left sidebar
- **Name:** `ROBOFLOW_API_KEY`
- **Value:** your (regenerated) Roboflow key
- Toggle **Notebook access** on

In [ ]:
!pip install -q roboflow ultralytics

In [ ]:
from roboflow import Roboflow

# Read the key from a Colab secret (falls back to a prompt if not on Colab)
try:
    from google.colab import userdata
    api_key = userdata.get('ROBOFLOW_API_KEY')
except Exception:
    import getpass
    api_key = getpass.getpass('Roboflow API key: ')

rf = Roboflow(api_key=api_key)
project = rf.workspace("roboflow-universe-projects").project("construction-site-safety")
version = project.version(30)
dataset = version.download("yolov8")
print("Downloaded to:", dataset.location)

## 2. Inspect classes

In [ ]:
import yaml
from pathlib import Path

root = Path(dataset.location)
with open(root / "data.yaml") as f:
    cfg = yaml.safe_load(f)

names = cfg["names"]
# Roboflow exports names as a list; normalize just in case
class_names = list(names.values()) if isinstance(names, dict) else list(names)

print("Number of classes:", len(class_names))
for i, n in enumerate(class_names):
    print(f"{i:2d}  {n}")

## 3. Count images per split

In [ ]:
for split in ["train", "valid", "test"]:
    img_dir = root / split / "images"
    if img_dir.exists():
        n = len(list(img_dir.glob("*.*")))
        print(f"{split:6s}: {n:5d} images")

## 4. Visualize annotated samples

Bounding boxes drawn from the YOLO labels. Look for: multiple people per frame,
lighting variety, and whether the NO-* classes (missing PPE) actually appear.
Red boxes = missing PPE (NO-*), green = present.

In [ ]:
import cv2, random
import matplotlib.pyplot as plt

def load_label(label_path):
    boxes = []
    if label_path.exists():
        for line in label_path.read_text().strip().splitlines():
            if not line:
                continue
            p = line.split()
            cls = int(p[0])
            cx, cy, w, h = map(float, p[1:5])
            boxes.append((cls, cx, cy, w, h))
    return boxes

def annotated(img_path):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]
    lbl = Path(str(img_path).replace("/images/", "/labels/")).with_suffix(".txt")
    for cls, cx, cy, w, h in load_label(lbl):
        x1, y1 = int((cx - w / 2) * W), int((cy - h / 2) * H)
        x2, y2 = int((cx + w / 2) * W), int((cy + h / 2) * H)
        miss = class_names[cls].upper().startswith("NO")
        color = (220, 30, 30) if miss else (30, 170, 60)
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        cv2.putText(img, class_names[cls], (x1, max(y1 - 5, 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
    return img

train_imgs = list((root / "train" / "images").glob("*.*"))
random.seed(0)
samples = random.sample(train_imgs, min(6, len(train_imgs)))

plt.figure(figsize=(16, 9))
for i, p in enumerate(samples):
    plt.subplot(2, 3, i + 1)
    plt.imshow(annotated(p))
    plt.axis("off")
plt.tight_layout()
plt.show()

## 5. Class balance

Counts every annotation in the training split. Imbalance here is normal
(Person and Hardhat dominate; NO-* and Mask are rarer) and affects how to read
per-class mAP later. Note this in the report - it is part of the grade.

In [ ]:
from collections import Counter

counts = Counter()
for lbl in (root / "train" / "labels").glob("*.txt"):
    for line in lbl.read_text().strip().splitlines():
        if line:
            counts[int(line.split()[0])] += 1

labels = [class_names[i] for i in range(len(class_names))]
values = [counts.get(i, 0) for i in range(len(class_names))]

plt.figure(figsize=(11, 5))
plt.bar(labels, values)
plt.xticks(rotation=45, ha="right")
plt.ylabel("Instance count (train)")
plt.title("Class balance - Construction Site Safety (train)")
plt.tight_layout()
plt.show()

print("Instances per class (train, descending):")
for n, v in sorted(zip(labels, values), key=lambda x: -x[1]):
    print(f"  {n:18s} {v}")

## Observations (fill in after running)

- Total images per split: _..._
- Most frequent classes: _..._
- Rarest classes (watch these in evaluation): _..._
- Imbalance strategy: rely on YOLOv8 built-in augmentation; consider more
  epochs and interpret per-class AP rather than only overall mAP.

**Next phase:** fine-tune `yolov8s.pt` on this dataset (training notebook).